In [1]:
# Clonar PnLCalib dentro del repo del TFG (en una carpeta externa para no mezclarlo con tu código)
from pathlib import Path
import subprocess, sys, os

REPO_ROOT = Path("../..").resolve()
PNL_DIR = REPO_ROOT / "external" / "PnLCalib"
PNL_DIR.parent.mkdir(parents=True, exist_ok=True)

if not PNL_DIR.exists():
    print("Clonando PnLCalib...")
    r = subprocess.run(["git", "clone", "https://github.com/mguti97/PnLCalib.git", str(PNL_DIR)],
                       capture_output=True, text=True)
    print(r.stdout); print(r.stderr)
else:
    print(f"PnLCalib ya existe en {PNL_DIR}")

# Ver qué contiene y, sobre todo, qué dependencias pide (no las adivinamos)
print("\nContenido del repositorio:")
for item in sorted(PNL_DIR.iterdir()):
    print(f"  {item.name}")

req = PNL_DIR / "requirements.txt"
if req.exists():
    print("\n=== requirements.txt (lo que PnLCalib necesita) ===")
    print(req.read_text())

Clonando PnLCalib...

Cloning into 'C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\external\PnLCalib'...


Contenido del repositorio:
  .git
  config
  eval_wp.py
  examples
  figures
  inference
  inference.py
  LICENSE
  model
  PnLCalib.yml
  README.md
  requirements.txt
  scripts
  sn_calibration
  train.py
  train_l.py
  utils

=== requirements.txt (lo que PnLCalib necesita) ===
coloredlogs==15.0.1
contourpy==1.3.0
cycler==0.12.1
filelock==3.17.0
flatbuffers==23.3.3
fonttools==4.55.8
fsspec==2025.2.0
hpack==4.0.0
humanfriendly==10.0
importlib_resources==6.5.2
Jinja2==3.1.5
kiwisolver==1.4.7
lsq-ellipse==2.2.1
MarkupSafe==3.0.2
matplotlib==3.9.4
mpmath==1.3.0
munkres==1.1.4
networkx==3.2.1
numpy==2.0.2
nvidia-cublas-cu12==12.1.3.1
nvidia-cuda-cupti-cu12==12.1.105
nvidia-cuda-nvrtc-cu12==12.1.105
nvidia-cuda-runtime-cu12==12.1.105
nvidia-cudnn-cu12==8.9.2.26
nvidia-cufft-cu12==11.0.2.54
nvidia-curand-cu12==10.3.2.106
nvidia-cusolver-cu12==11.4.5.107
nvidia-cusparse-cu12==12.1.0.106
nv

In [2]:
# Leer el inference.py para saber sus argumentos exactos y qué produce
inf_path = PNL_DIR / "inference.py"
print(inf_path.read_text()[:4000])

import cv2
import yaml
import torch
import argparse
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as T
import torchvision.transforms.functional as f

from tqdm import tqdm
from PIL import Image
from matplotlib.patches import Polygon

from model.cls_hrnet import get_cls_net
from model.cls_hrnet_l import get_cls_net as get_cls_net_l

from utils.utils_calib import FramebyFrameCalib, pan_tilt_roll_to_orientation
from utils.utils_heatmap import get_keypoints_from_heatmap_batch_maxpool, get_keypoints_from_heatmap_batch_maxpool_l, \
    complete_keypoints, coords_to_dict


lines_coords = [[[0., 54.16, 0.], [16.5, 54.16, 0.]],
                [[16.5, 13.84, 0.], [16.5, 54.16, 0.]],
                [[16.5, 13.84, 0.], [0., 13.84, 0.]],
                [[88.5, 54.16, 0.], [105., 54.16, 0.]],
                [[88.5, 13.84, 0.], [88.5, 54.16, 0.]],
                [[88.5, 13.84, 0.], [105., 13.84, 0.]],
                [[0., 37.66, -2.44], [0., 30.34, -2.44]],
  

In [4]:
import subprocess, sys

# Instalar en el MISMO Python del notebook (.venv), sin tocar torch.
# shapely y lsq-ellipse son las que PnLCalib necesita y no tienes.
faltan = ["shapely", "lsq-ellipse", "coloredlogs", "humanfriendly"]

for paquete in faltan:
    print(f"Instalando {paquete}...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", paquete],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f"  OK")
    else:
        print(f"  ERROR:\n{r.stderr[-500:]}")

# Verificar que ahora sí importan, en este mismo Python
print("\nVerificación de imports:")
for mod in ["shapely", "ellipse", "coloredlogs", "humanfriendly"]:
    try:
        __import__(mod)
        print(f"  {mod}: OK")
    except ImportError as e:
        print(f"  {mod}: FALTA ({e})")

Instalando shapely...
  OK
Instalando lsq-ellipse...
  OK
Instalando coloredlogs...
  OK
Instalando humanfriendly...
  OK

Verificación de imports:
  shapely: OK
  ellipse: OK
  coloredlogs: OK
  humanfriendly: OK


In [6]:
import urllib.request
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
PNL_DIR = REPO_ROOT / "external" / "PnLCalib"
weights_dir = PNL_DIR / "weights"
weights_dir.mkdir(exist_ok=True)

pesos = {
    "SV_kp":    "https://github.com/mguti97/PnLCalib/releases/download/v1.0.0/SV_kp",
    "SV_lines": "https://github.com/mguti97/PnLCalib/releases/download/v1.0.0/SV_lines",
}

for nombre, url in pesos.items():
    dest = weights_dir / nombre
    if dest.exists() and dest.stat().st_size > 1_000_000:
        print(f"{nombre} ya está ({dest.stat().st_size//1024//1024} MB)")
        continue
    print(f"Descargando {nombre} desde {url} ...")
    try:
        urllib.request.urlretrieve(url, dest)
        size_mb = dest.stat().st_size // 1024 // 1024
        print(f"  guardado: {size_mb} MB")
        if size_mb < 1:
            print(f"  ⚠️ AVISO: el archivo es muy pequeño ({dest.stat().st_size} bytes), "
                  f"puede que la descarga fallara o sea una página de error, no el modelo")
    except Exception as e:
        print(f"  ERROR descargando: {e}")

# Verificar que están y con tamaño razonable
print("\nEstado de los pesos:")
for nombre in pesos:
    dest = weights_dir / nombre
    if dest.exists():
        print(f"  {dest}  ->  {dest.stat().st_size//1024//1024} MB")
    else:
        print(f"  {dest}  ->  NO EXISTE")

Descargando SV_kp desde https://github.com/mguti97/PnLCalib/releases/download/v1.0.0/SV_kp ...
  guardado: 252 MB
Descargando SV_lines desde https://github.com/mguti97/PnLCalib/releases/download/v1.0.0/SV_lines ...
  guardado: 252 MB

Estado de los pesos:
  C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\external\PnLCalib\weights\SV_kp  ->  252 MB
  C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\external\PnLCalib\weights\SV_lines  ->  252 MB


In [8]:
import subprocess, sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
PNL_DIR = REPO_ROOT / "external" / "PnLCalib"
VIDEO_IN = REPO_ROOT / "data" / "raw" / "A1606b0e6_0 (62).mp4"
VIDEO_OUT = REPO_ROOT / "outputs" / "pnlcalib_validacion_A1606.mp4"
VIDEO_OUT.parent.mkdir(parents=True, exist_ok=True)

# Volcar la salida a un archivo de log en vez de capturarla como texto (evita el error cp1252 de Windows)
log_path = REPO_ROOT / "outputs" / "pnlcalib_log.txt"

cmd = [
    sys.executable, "inference.py",
    "--weights_kp", "weights/SV_kp",
    "--weights_line", "weights/SV_lines",
    "--pnl_refine",
    "--input_path", str(VIDEO_IN),
    "--input_type", "video",
    "--save_path", str(VIDEO_OUT),
]
print("Ejecutando PnLCalib (la salida va al log; esto tarda varios minutos)...")
print(" ".join(cmd))

with open(log_path, "w", encoding="utf-8", errors="replace") as logf:
    proc = subprocess.run(cmd, cwd=str(PNL_DIR), stdout=logf, stderr=subprocess.STDOUT)

print(f"\nCódigo de salida: {proc.returncode}  (0 = OK)")
print(f"¿Vídeo generado? {VIDEO_OUT.exists()}")
if VIDEO_OUT.exists():
    print(f"  tamaño: {VIDEO_OUT.stat().st_size//1024//1024} MB")

# Mostrar las últimas líneas del log, leídas con tolerancia a caracteres raros
print("\n--- Últimas líneas del log ---")
print(log_path.read_text(encoding="utf-8", errors="replace")[-1500:])

Ejecutando PnLCalib (la salida va al log; esto tarda varios minutos)...
C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\.venv\Scripts\python.exe inference.py --weights_kp weights/SV_kp --weights_line weights/SV_lines --pnl_refine --input_path C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\raw\A1606b0e6_0 (62).mp4 --input_type video --save_path C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\outputs\pnlcalib_validacion_A1606.mp4

Código de salida: 0  (0 = OK)
¿Vídeo generado? True
  tamaño: 58 MB

--- Últimas líneas del log ---
100%|██████████| 749/749 [06:35<00:00,  1.89it/s]

